# Three engines, one model, one card — metrics, memory, and where every step's time goes

Tesla T4, Qwen3-0.6B FP16, identical pre-tokenized prompts, EOS ignored, greedy. Three engines,
each in its own process:

| engine | what it is |
|---|---|
| `naive_hf` | stock Hugging Face `generate`: DynamicCache, static left-padded batches, no paging |
| `repo_asis` / `repo_no_graphs` | this engine at its defaults, and with CUDA graphs off |
| `vllm` | vLLM at the **same KV capacity** (16,384 tokens via `kv_cache_memory_bytes`), same concurrency cap |

**Part A — metrics and memory.** TTFT on a 512-token prompt, TPOT, single-stream and batched
throughput at 1/4/8/16, startup, NVML resident and peak. For this engine the memory is broken
into stages — CUDA context → weights → KV pool → warm-up → `empty_cache` — which gives the
**private CUDA-graph pool** and the **graph executables outside PyTorch's allocator** as two
separate numbers. These are the post-fix figures the reachable-shape change was supposed to
move and that were never recorded end to end; the `no_graphs` arm is the control.

**Part B — where a step's device time goes, per engine, by transformer sub-block.** A
`torch.profiler` run over a 512-token prefill and over decode at batch 8 with ~512-token
contexts. CUDA kernel events are classified by name into attention, norm, RoPE, activation,
sampling, memcpy and elementwise; GEMM time is apportioned to **FFN / attention projections /
lm_head** from the aten-level `mm`/`addmm` shapes, which are identical across engines because
the model is. Decode is isolated the same way for all three — a 33-token run minus a 1-token
run, divided by 32 — so vLLM's scheduler and this engine's fused steps are measured by the
same rule. For this engine one exact `step()` is profiled too, as a cross-check.

Also reported per step: **kernel launches vs `cudaGraphLaunch` calls** — the CUDA-graph story
in one number.

vLLM runs in-process (`VLLM_ENABLE_V1_MULTIPROCESSING=0`) so the profiler can see its
kernels; its speed here is a slight underestimate of its multiprocess best (part 1 has those).

**Runtime:** ~35–45 min total. The vLLM install adds ~5 min on a fresh runtime.

In [ ]:
# ---------------- configuration ----------------
MODEL = "Qwen/Qwen3-0.6B"
KV_TOKENS = 16384        # KV capacity for this engine and vLLM alike (1.75 GiB at 112 KiB/token)
MAX_ACTIVE = 16
MAX_MODEL_LEN = 1024     # vLLM's; this engine is sized by the pool
GEN_TOKENS = 128
RUNS = 3

RUN_VLLM = True
VLLM_SPEC = "vllm"       # pin e.g. "vllm==0.30.0" to match part 1
REPO_REF = "main"
REPO_ZIP = None          # e.g. "/content/full-inference-engine.zip" to run an unpushed tree
TIMEOUT_S = 2400

WORKDIR = "/content/bench4"
REPO_DIR = "/content/full-inference-engine"
VLLM_PY = "/content/vllm-env/bin/python"
import os
for d in ("results", "logs"):
    os.makedirs(f"{WORKDIR}/{d}", exist_ok=True)

## 1. Setup

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
if REPO_ZIP:
    import glob as _glob, shutil, zipfile
    shutil.rmtree(REPO_DIR, ignore_errors=True); os.makedirs(REPO_DIR, exist_ok=True)
    with zipfile.ZipFile(REPO_ZIP) as archive:
        archive.extractall(REPO_DIR)
    if not os.path.isdir(f"{REPO_DIR}/engine"):
        inner = [d for d in _glob.glob(f"{REPO_DIR}/*") if os.path.isdir(f"{d}/engine")]
        for item in _glob.glob(f"{inner[0]}/*") + _glob.glob(f"{inner[0]}/.??*"):
            shutil.move(item, REPO_DIR)
    print("engine source from", REPO_ZIP)
else:
    !test -d {REPO_DIR} || git clone -q https://github.com/Vaibhav7711/full-inference-engine.git {REPO_DIR}
    !cd {REPO_DIR} && git fetch -q --all --tags && git checkout -q {REPO_REF} && (git pull -q --ff-only 2>/dev/null || true)
    !cd {REPO_DIR} && git log -1 --format='repo commit: %h %ad %s' --date=short
!python -c "import engine" 2>/dev/null || (pip install -q -r {REPO_DIR}/requirements-colab.txt nvidia-ml-py && pip install -q -e {REPO_DIR})
!pip install -q nvidia-ml-py uv
if RUN_VLLM:
    !test -x {VLLM_PY} || (uv venv -q /content/vllm-env --python python3 && uv pip install -q --python {VLLM_PY} "{VLLM_SPEC}" nvidia-ml-py --torch-backend=auto)
    !{VLLM_PY} -c "import vllm, torch; print('vLLM', vllm.__version__, '| torch', torch.__version__)"
import torch; print("repo env torch", torch.__version__)

## 2. Workers

In [ ]:
%%writefile /content/bench4/bench4_common.py
"""Shared helpers for the three-engine comparison workers (stdlib + torch + pynvml).

Two things live here so every engine is measured the same way: the NVML memory tracker,
and the profiler classifier that turns a torch.profiler run into "where did the step's
device time go" by transformer sub-block. The classifier works on CUDA kernel events by
name, which is engine-agnostic - a GEMM is a GEMM whether cuBLAS, CUTLASS or Triton
launched it - and then splits GEMM time between FFN, attention projections and lm_head
using the aten-level `mm`/`addmm` input shapes, which are the same for every engine
because the model is the same.
"""
from __future__ import annotations

import re
import statistics
import threading
import time


class GpuMemoryTracker:
    """Device-level GPU memory via NVML, polled in a background thread."""

    def __init__(self, interval_s: float = 0.02):
        import pynvml

        self._nvml = pynvml
        pynvml.nvmlInit()
        self._handle = pynvml.nvmlDeviceGetHandleByIndex(0)
        name = pynvml.nvmlDeviceGetName(self._handle)
        self.gpu_name = name.decode() if isinstance(name, bytes) else name
        self.baseline_mib = self.used_mib()
        self.peak_mib = self.baseline_mib
        self._interval = interval_s
        self._stop = threading.Event()
        self._thread = threading.Thread(target=self._poll, daemon=True)

    def used_mib(self) -> float:
        return self._nvml.nvmlDeviceGetMemoryInfo(self._handle).used / 2**20

    def above_baseline(self) -> float:
        return self.used_mib() - self.baseline_mib

    def reset_peak(self) -> None:
        self.peak_mib = self.used_mib()

    def peak_above_baseline(self) -> float:
        return max(self.peak_mib, self.used_mib()) - self.baseline_mib

    def _poll(self) -> None:
        while not self._stop.is_set():
            self.peak_mib = max(self.peak_mib, self.used_mib())
            time.sleep(self._interval)

    def start(self) -> "GpuMemoryTracker":
        self._thread.start()
        return self

    def stop(self) -> None:
        self._stop.set()
        self._thread.join(timeout=2)
        self.peak_mib = max(self.peak_mib, self.used_mib())


def median_of(fn, runs: int, warmup: int = 1):
    for _ in range(warmup):
        fn()
    return statistics.median(fn() for _ in range(runs))


# ---------------------------------------------------------------------------
# Profiler classification
# ---------------------------------------------------------------------------

# Kernel-name patterns, checked in order; first match wins.
KERNEL_CLASSES = [
    ("attention", re.compile(
        r"paged_decode|paged_prefill|tiled_paged|flash|fmha|attn|attention|sdpa|"
        r"mem_efficient|scaled_dot|write_decode_kv|write_prefill_kv|kv_write|"
        r"reshape_and_cache|paged_attention|softmax", re.I)),
    ("gemm", re.compile(r"gemm|cutlass|gemv|cublas|Lt|sgemm|hgemm|igemm|mma|xmma|wgmma|"
                        r"ampere|turing|volta|sm75|sm80|sm89|matmul|splitK|dot", re.I)),
    ("norm", re.compile(r"rms|layer_norm|layernorm|norm", re.I)),
    ("rope", re.compile(r"rope|rotary", re.I)),
    ("activation", re.compile(r"silu|swiglu|gelu|act_and_mul|sigmoid|mul_kernel", re.I)),
    ("sampling", re.compile(r"argmax|topk|top_k|multinomial|sort|radix|cumsum|sampl", re.I)),
    ("embedding", re.compile(r"embedding|index_select|gather", re.I)),
    ("memcpy", re.compile(r"memcpy|memset|copy_|Memcpy|Memset|elementwise_kernel.*copy", re.I)),
    ("elementwise", re.compile(r"elementwise|vectorized|unrolled|fill|cat|add|arange|"
                               r"where|clamp|compare|reduce|binary|unary", re.I)),
]

VOCAB = 151936


def _gemm_class_from_shapes(shapes, hidden: int = 1024, intermediate: int = 3072) -> str:
    """Which GEMM an aten::mm/addmm is, from the model's own dimensions."""
    try:
        dims = [tuple(s) for s in shapes if isinstance(s, (list, tuple)) and len(s) == 2]
        if not dims:
            return "gemm_other"
        # Second operand is the weight (or its transpose): [K, N] or [N, K].
        a, b = dims[0], dims[1] if len(dims) > 1 else dims[0]
        candidates = set(a) | set(b)
        if VOCAB in candidates:
            return "lm_head"
        if intermediate in candidates or 2 * intermediate in candidates:
            return "ffn"
        return "attn_proj"
    except Exception:
        return "gemm_other"


def classify_profile(prof, hidden: int = 1024, intermediate: int = 3072) -> dict:
    """Device time by sub-block, kernel launch counts, and the GEMM split, from a profile.

    Only CUDA-side events (kernels, memcpy, memset) are summed, so nothing is counted
    twice. GEMM kernel time is then apportioned by the aten-level shapes.
    """
    from torch.autograd import DeviceType

    averages = prof.key_averages(group_by_input_shape=True)
    by_class: dict[str, float] = {}
    kernel_launches = 0
    graph_launches = 0
    gemm_by_role: dict[str, float] = {}
    for event in averages:
        name = event.key
        if name in ("cudaLaunchKernel", "cuLaunchKernel"):
            kernel_launches += int(event.count)
            continue
        if name in ("cudaGraphLaunch", "cuGraphLaunch"):
            graph_launches += int(event.count)
            continue
        is_device = getattr(event, "device_type", None) == DeviceType.CUDA
        device_us = float(getattr(event, "self_device_time_total",
                                  getattr(event, "self_cuda_time_total", 0.0)) or 0.0)
        if name in ("aten::mm", "aten::addmm", "aten::bmm", "aten::matmul", "aten::linear"):
            # Apportioning only: total (not self) device time of the GEMM op by shape.
            total_us = float(getattr(event, "device_time_total",
                                     getattr(event, "cuda_time_total", 0.0)) or 0.0)
            if name != "aten::linear" and total_us > 0:
                role = _gemm_class_from_shapes(getattr(event, "input_shapes", []) or [],
                                               hidden, intermediate)
                gemm_by_role[role] = gemm_by_role.get(role, 0.0) + total_us
            continue
        if not is_device or device_us <= 0:
            continue
        for label, pattern in KERNEL_CLASSES:
            if pattern.search(name):
                by_class[label] = by_class.get(label, 0.0) + device_us
                break
        else:
            by_class["other"] = by_class.get("other", 0.0) + device_us

    gemm_total = by_class.pop("gemm", 0.0)
    role_total = sum(gemm_by_role.values())
    if gemm_total > 0 and role_total > 0:
        for role, share in gemm_by_role.items():
            by_class[f"gemm_{role}"] = gemm_total * share / role_total
    elif gemm_total > 0:
        by_class["gemm_unattributed"] = gemm_total
    total_ms = sum(by_class.values()) / 1000.0
    return {
        "device_ms_by_class": {k: v / 1000.0 for k, v in sorted(by_class.items())},
        "device_ms_total": total_ms,
        "kernel_launches": kernel_launches,
        "graph_launches": graph_launches,
    }


def subtract_profiles(long: dict, short: dict, steps: int) -> dict:
    """Per-decode-step attribution: (33-token run - 1-token run) / 32, class by class."""
    classes = set(long["device_ms_by_class"]) | set(short["device_ms_by_class"])
    per_step = {
        c: max(0.0, (long["device_ms_by_class"].get(c, 0.0)
                     - short["device_ms_by_class"].get(c, 0.0)) / steps)
        for c in sorted(classes)
    }
    return {
        "device_ms_by_class": per_step,
        "device_ms_total": sum(per_step.values()),
        "kernel_launches": max(0, (long["kernel_launches"] - short["kernel_launches"]) // steps),
        "graph_launches": max(0, (long["graph_launches"] - short["graph_launches"]) // steps),
    }


def profile_call(fn):
    """Run `fn` under torch.profiler and return the classified result and wall ms."""
    import torch
    from torch.profiler import ProfilerActivity, profile

    torch.cuda.synchronize()
    fn()                                       # one warm call outside the profile
    torch.cuda.synchronize()
    start = time.perf_counter()
    with profile(activities=(ProfilerActivity.CPU, ProfilerActivity.CUDA),
                 record_shapes=True) as prof:
        fn()
        torch.cuda.synchronize()
    wall_ms = (time.perf_counter() - start) * 1000
    result = classify_profile(prof)
    result["wall_ms"] = wall_ms
    return result

In [ ]:
%%writefile /content/bench4/bench4_repo.py
"""This engine: memory by stage (the post-fix graph numbers), speed, and per-block profile."""
from __future__ import annotations

import argparse
import json
import time
import traceback

from bench4_common import GpuMemoryTracker, median_of, profile_call, subtract_profiles


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--model", required=True)
    p.add_argument("--variant", default="asis", choices=("asis", "no_graphs"))
    p.add_argument("--kv-tokens", type=int, default=16384)
    p.add_argument("--max-active", type=int, default=16)
    p.add_argument("--gen-tokens", type=int, default=128)
    p.add_argument("--runs", type=int, default=3)
    p.add_argument("--out", required=True)
    args = p.parse_args()
    tracker = GpuMemoryTracker().start()
    stages = {}
    try:
        import torch
        from engine.batching.continuous_batching import ContinuousBatchingEngine
        from engine.model import load_model
        from engine.runtime import GenerationRequest
        from engine.runtime.sampling import SamplingParams

        def snap(name):
            torch.cuda.synchronize()
            stages[name] = {"nvml_mib": round(tracker.above_baseline()),
                            "allocated_mib": round(torch.cuda.memory_allocated() / 2**20),
                            "reserved_mib": round(torch.cuda.memory_reserved() / 2**20)}

        t_start = time.perf_counter()
        torch.zeros(1, device="cuda")
        snap("1_cuda_context")
        loaded = load_model(args.model, dtype="float16")
        snap("2_weights")
        block = 16
        kwargs = {}
        if args.variant != "no_graphs":
            kwargs["cuda_graph_batch_sizes"] = tuple(b for b in (1, 2, 4, 8, 16) if b <= args.max_active)
        engine = ContinuousBatchingEngine(
            loaded.model, loaded.tokenizer, loaded.device, max_active=args.max_active,
            num_blocks=args.kv_tokens // block, block_size=block, prefix_cache_blocks=0, **kwargs)
        snap("3_kv_pool_and_buffers")
        t0 = time.perf_counter()
        summary = engine.warmup()
        torch.cuda.synchronize()
        warmup_s = time.perf_counter() - t0
        snap("4_after_warmup")
        torch.cuda.empty_cache()
        snap("5_after_empty_cache")
        startup_s = time.perf_counter() - t_start
        resident = tracker.above_baseline()

        # What warm-up captured and the widest shape it gathered - the numbers the
        # reachable-shape fix changed and that were never recorded end to end.
        kv_per_token = 2 * engine.num_kv_heads * engine.head_dim * 2
        captured = [(r, c) for r, _a, c in engine._prefill_graphs] + \
                   [(pr, c) for _d, pr, _a, c, _bn, _nw in engine._fused_graphs]
        widest = max(captured, key=lambda rc: rc[0] * rc[1], default=(0, 0))
        geometry = {
            "pool_tokens": engine.key_pool[0].shape[0] * block,
            "graphs": summary.get("graphs"), "prefill_graphs": summary.get("prefill_graphs"),
            "fused_graphs": summary.get("fused_graphs"),
            "prefill_graph_unsupported": summary.get("prefill_graph_unsupported"),
            "widest_captured_rows_x_context": list(widest),
            "largest_gather_mib_per_layer": round(widest[0] * widest[1] * kv_per_token / 2**20, 1),
            "dummy_row_tokens": len(engine._graph_dummy_blocks) * block,
        }

        vocab = int(loaded.model.config.vocab_size)
        gen = torch.Generator().manual_seed(0)
        rand_ids = lambda n: torch.randint(1000, vocab - 1000, (n,), generator=gen).tolist()
        sp = SamplingParams(ignore_eos=True)

        def run(prompt_lists, n):
            engine.reset()
            reqs = [GenerationRequest(request_id=f"r{i}", prompt_token_count=len(ids),
                                      max_new_tokens=n, prompt_token_ids=list(ids), sampling=sp)
                    for i, ids in enumerate(prompt_lists)]
            torch.cuda.synchronize()
            t = time.perf_counter()
            for r in reqs:
                if engine.submit(r) is False:
                    raise RuntimeError("rejected")
            while engine.has_unfinished_requests:
                engine.step()
            torch.cuda.synchronize()
            return (time.perf_counter() - t) * 1000

        tracker.reset_peak()
        single = [rand_ids(512)]
        ttft = median_of(lambda: run(single, 1), args.runs)
        e2e = median_of(lambda: run(single, args.gen_tokens), args.runs)
        batched = {}
        for b in (1, 4, 8, 16):
            if b > args.max_active:
                continue
            prompts = [rand_ids(256) for _ in range(b)]
            ms = median_of(lambda: run(prompts, args.gen_tokens), args.runs)
            batched[str(b)] = b * args.gen_tokens / (ms / 1000)
        speed = {"ttft_512_ms": ttft, "tpot_ms": (e2e - ttft) / (args.gen_tokens - 1),
                 "single_stream_tok_s": (args.gen_tokens - 1) / ((e2e - ttft) / 1000),
                 "batched_tok_s": batched, "lazy_graph_captures": engine.lazy_graph_captures}
        peak_speed = tracker.peak_above_baseline()

        # ---- per-block profiles -------------------------------------------------------
        eight = [rand_ids(512) for _ in range(8)]
        prefill = profile_call(lambda: run(single, 1))
        long = profile_call(lambda: run(eight, 33))
        short = profile_call(lambda: run(eight, 1))
        decode = subtract_profiles(long, short, 32)
        decode["wall_ms"] = (long["wall_ms"] - short["wall_ms"]) / 32

        # Exact single decode step at batch 8, ~512 context, as a cross-check.
        engine.reset()
        reqs = [GenerationRequest(request_id=f"s{i}", prompt_token_count=512, max_new_tokens=64,
                                  prompt_token_ids=list(ids), sampling=sp) for i, ids in enumerate(eight)]
        for r in reqs:
            engine.submit(r)
        for _ in range(2000):
            engine.step()
            if engine.stats_snapshot()["decode_batch"] == 8 and engine.last_step_prefill_tokens == 0:
                break
        one_step = profile_call(lambda: engine.step())
        tracker.stop()
        w, k = stages["3_kv_pool_and_buffers"], stages["4_after_warmup"]
        result = {
            "ok": True, "engine": f"repo_{args.variant}", "model": args.model, "gpu": tracker.gpu_name,
            "versions": {"torch": torch.__version__, "triton": __import__("triton").__version__},
            "startup_s": round(startup_s, 1), "warmup_s": round(warmup_s, 1),
            "stages": stages, "graph_geometry": geometry,
            "attribution_mib": {
                "cuda_context": stages["1_cuda_context"]["nvml_mib"],
                "weights": stages["2_weights"]["allocated_mib"] - stages["1_cuda_context"]["allocated_mib"],
                "kv_pool_and_buffers": w["allocated_mib"] - stages["2_weights"]["allocated_mib"],
                "graph_pool": k["reserved_mib"] - w["reserved_mib"] - (k["allocated_mib"] - w["allocated_mib"]),
                "outside_allocator_warmup": (k["nvml_mib"] - w["nvml_mib"]) - (k["reserved_mib"] - w["reserved_mib"]),
            },
            "resident_mib": round(resident), "peak_mib": round(peak_speed),
            "speed": speed,
            "profile": {"prefill_512": prefill, "decode_step_b8_ctx512": decode,
                        "decode_one_step_exact": one_step},
            "config": {"kv_tokens": args.kv_tokens, "max_active": args.max_active,
                       "prefill_attention": engine.prefill_attention,
                       "decode_attention": engine.decode_attention},
        }
    except Exception as error:
        tracker.stop()
        result = {"ok": False, "engine": f"repo_{args.variant}", "model": args.model,
                  "error": repr(error), "traceback": traceback.format_exc()[-5000:], "stages": stages}
    with open(args.out, "w") as f:
        json.dump(result, f, indent=2, default=str)
    print("saved", args.out, flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/bench4/bench4_naive.py
"""Naive baseline: stock Hugging Face `generate`, DynamicCache, padding for batches."""
from __future__ import annotations

import argparse
import json
import time
import traceback

from bench4_common import GpuMemoryTracker, median_of, profile_call, subtract_profiles


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--model", required=True)
    p.add_argument("--gen-tokens", type=int, default=128)
    p.add_argument("--runs", type=int, default=3)
    p.add_argument("--out", required=True)
    args = p.parse_args()
    tracker = GpuMemoryTracker().start()
    stages = {}
    try:
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer

        def snap(name):
            torch.cuda.synchronize()
            stages[name] = {"nvml_mib": round(tracker.above_baseline()),
                            "allocated_mib": round(torch.cuda.memory_allocated() / 2**20),
                            "reserved_mib": round(torch.cuda.memory_reserved() / 2**20)}

        t_start = time.perf_counter()
        torch.zeros(1, device="cuda")
        snap("1_cuda_context")
        tok = AutoTokenizer.from_pretrained(args.model)
        model = AutoModelForCausalLM.from_pretrained(args.model, dtype=torch.float16).to("cuda").eval()
        snap("2_weights")
        startup_s = time.perf_counter() - t_start
        vocab = int(model.config.vocab_size)
        gen = torch.Generator().manual_seed(0)
        rand_ids = lambda n: torch.randint(1000, vocab - 1000, (n,), generator=gen).tolist()
        pad = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id

        @torch.inference_mode()
        def run(prompt_lists, n):
            # Left-pad to a batch; attention_mask carries the padding. Length is forced
            # with min_new_tokens so EOS cannot shorten a run, matching ignore_eos.
            width = max(len(x) for x in prompt_lists)
            ids = torch.full((len(prompt_lists), width), pad, dtype=torch.long)
            mask = torch.zeros_like(ids)
            for i, x in enumerate(prompt_lists):
                ids[i, width - len(x):] = torch.tensor(x)
                mask[i, width - len(x):] = 1
            ids, mask = ids.cuda(), mask.cuda()
            torch.cuda.synchronize()
            t = time.perf_counter()
            model.generate(input_ids=ids, attention_mask=mask, max_new_tokens=n, min_new_tokens=n,
                           do_sample=False, pad_token_id=pad, use_cache=True)
            torch.cuda.synchronize()
            return (time.perf_counter() - t) * 1000

        tracker.reset_peak()
        single = [rand_ids(512)]
        ttft = median_of(lambda: run(single, 1), args.runs)
        e2e = median_of(lambda: run(single, args.gen_tokens), args.runs)
        batched = {}
        for b in (1, 4, 8, 16):
            prompts = [rand_ids(256) for _ in range(b)]
            ms = median_of(lambda: run(prompts, args.gen_tokens), args.runs)
            batched[str(b)] = b * args.gen_tokens / (ms / 1000)
        speed = {"ttft_512_ms": ttft, "tpot_ms": (e2e - ttft) / (args.gen_tokens - 1),
                 "single_stream_tok_s": (args.gen_tokens - 1) / ((e2e - ttft) / 1000),
                 "batched_tok_s": batched}
        peak = tracker.peak_above_baseline()
        torch.cuda.empty_cache()
        resident = tracker.above_baseline()

        eight = [rand_ids(512) for _ in range(8)]
        prefill = profile_call(lambda: run(single, 1))
        long = profile_call(lambda: run(eight, 33))
        short = profile_call(lambda: run(eight, 1))
        decode = subtract_profiles(long, short, 32)
        decode["wall_ms"] = (long["wall_ms"] - short["wall_ms"]) / 32
        tracker.stop()
        result = {
            "ok": True, "engine": "naive_hf", "model": args.model, "gpu": tracker.gpu_name,
            "versions": {"torch": torch.__version__,
                         "transformers": __import__("transformers").__version__},
            "startup_s": round(startup_s, 1), "stages": stages,
            "attribution_mib": {
                "cuda_context": stages["1_cuda_context"]["nvml_mib"],
                "weights": stages["2_weights"]["allocated_mib"] - stages["1_cuda_context"]["allocated_mib"],
            },
            "resident_mib": round(resident), "peak_mib": round(peak), "speed": speed,
            "profile": {"prefill_512": prefill, "decode_step_b8_ctx512": decode},
            "config": {"kv": "DynamicCache, unbounded", "batching": "static, left-padded"},
        }
    except Exception as error:
        tracker.stop()
        result = {"ok": False, "engine": "naive_hf", "model": args.model, "error": repr(error),
                  "traceback": traceback.format_exc()[-5000:], "stages": stages}
    with open(args.out, "w") as f:
        json.dump(result, f, indent=2, default=str)
    print("saved", args.out, flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/bench4/bench4_vllm.py
"""vLLM at matched KV capacity, run in-process so the profiler sees its kernels.

`VLLM_ENABLE_V1_MULTIPROCESSING=0` is set before vLLM is imported: in V1 the engine core
normally lives in a separate process, where a torch.profiler in this one would see
nothing. In-process costs vLLM a little scheduling overlap, so the speed numbers here are
a slight underestimate of its best; the part-1 notebook has the multiprocess figures.
"""
from __future__ import annotations

import argparse
import json
import os
import time
import traceback

os.environ.setdefault("VLLM_ENABLE_V1_MULTIPROCESSING", "0")

from bench4_common import GpuMemoryTracker, median_of, profile_call, subtract_profiles


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--model", required=True)
    p.add_argument("--kv-bytes", type=int, required=True)
    p.add_argument("--max-model-len", type=int, default=1024)
    p.add_argument("--max-active", type=int, default=16)
    p.add_argument("--gen-tokens", type=int, default=128)
    p.add_argument("--runs", type=int, default=3)
    p.add_argument("--gpu-memory-utilization", type=float, default=0.90)
    p.add_argument("--out", required=True)
    args = p.parse_args()
    tracker = GpuMemoryTracker().start()
    try:
        import torch
        import vllm
        from transformers import AutoConfig
        from vllm import LLM, SamplingParams

        t0 = time.perf_counter()
        kwargs = dict(model=args.model, dtype="half", max_model_len=args.max_model_len,
                      max_num_seqs=args.max_active, enable_prefix_caching=False,
                      gpu_memory_utilization=args.gpu_memory_utilization,
                      kv_cache_memory_bytes=args.kv_bytes, seed=0, disable_log_stats=True)
        kv_matched = True
        try:
            llm = LLM(**kwargs)
        except TypeError:
            kwargs.pop("kv_cache_memory_bytes")
            kv_matched = False
            llm = LLM(**kwargs)
        startup_s = time.perf_counter() - t0
        after_load = tracker.above_baseline()

        vocab = int(AutoConfig.from_pretrained(args.model).vocab_size)
        import random
        rng = random.Random(0)
        rand_ids = lambda n: [rng.randrange(1000, vocab - 1000) for _ in range(n)]

        def run(prompt_lists, n):
            params = SamplingParams(temperature=0.0, max_tokens=n, ignore_eos=True)
            prompts = [{"prompt_token_ids": list(ids)} for ids in prompt_lists]
            t = time.perf_counter()
            llm.generate(prompts, params, use_tqdm=False)
            return (time.perf_counter() - t) * 1000

        tracker.reset_peak()
        single = [rand_ids(512)]
        ttft = median_of(lambda: run(single, 1), args.runs)
        e2e = median_of(lambda: run(single, args.gen_tokens), args.runs)
        batched = {}
        for b in (1, 4, 8, 16):
            if b > args.max_active:
                continue
            prompts = [rand_ids(256) for _ in range(b)]
            ms = median_of(lambda: run(prompts, args.gen_tokens), args.runs)
            batched[str(b)] = b * args.gen_tokens / (ms / 1000)
        speed = {"ttft_512_ms": ttft, "tpot_ms": (e2e - ttft) / (args.gen_tokens - 1),
                 "single_stream_tok_s": (args.gen_tokens - 1) / ((e2e - ttft) / 1000),
                 "batched_tok_s": batched}
        peak = tracker.peak_above_baseline()
        resident = tracker.above_baseline()

        eight = [rand_ids(512) for _ in range(8)]
        prefill = profile_call(lambda: run(single, 1))
        long = profile_call(lambda: run(eight, 33))
        short = profile_call(lambda: run(eight, 1))
        decode = subtract_profiles(long, short, 32)
        decode["wall_ms"] = (long["wall_ms"] - short["wall_ms"]) / 32
        tracker.stop()
        result = {
            "ok": True, "engine": "vllm", "model": args.model, "gpu": tracker.gpu_name,
            "versions": {"vllm": vllm.__version__, "torch": torch.__version__},
            "startup_s": round(startup_s, 1),
            "resident_after_load_mib": round(after_load), "resident_mib": round(resident),
            "peak_mib": round(peak), "speed": speed,
            "profile": {"prefill_512": prefill, "decode_step_b8_ctx512": decode},
            "config": {"kv_bytes": args.kv_bytes, "kv_capacity_matched": kv_matched,
                       "max_model_len": args.max_model_len, "in_process": True,
                       "cuda_graphs": "default (on)"},
        }
    except Exception as error:
        tracker.stop()
        result = {"ok": False, "engine": "vllm", "model": args.model, "error": repr(error),
                  "traceback": traceback.format_exc()[-5000:]}
    with open(args.out, "w") as f:
        json.dump(result, f, indent=2, default=str)
    print("saved", args.out, flush=True)


if __name__ == "__main__":
    main()

In [ ]:
import json, subprocess, sys, time

def gpu_used_mib():
    out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                         capture_output=True, text=True).stdout
    return float(out.strip().splitlines()[0])

def wait_for_idle_gpu(limit_mib=800, timeout_s=90):
    start = time.time()
    while gpu_used_mib() > limit_mib and time.time() - start < timeout_s:
        time.sleep(2)

def run_proc(cmd, log, out, env_repo):
    env = dict(os.environ, PYTHONPATH=WORKDIR + (f":{REPO_DIR}" if env_repo else ""),
               TOKENIZERS_PARALLELISM="false")
    wait_for_idle_gpu()
    if os.path.exists(out):
        os.remove(out)
    t0 = time.time()
    with open(log, "w") as logf:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                                env=env, cwd=REPO_DIR if env_repo else WORKDIR)
        try:
            for line in proc.stdout:
                logf.write(line)
                if line.startswith("saved") or "Error" in line:
                    print("  " + line.rstrip(), flush=True)
            proc.wait(timeout=TIMEOUT_S)
        except subprocess.TimeoutExpired:
            proc.kill(); print("  timed out")
    if not os.path.exists(out):
        json.dump({"ok": False, "error": f"exit code {proc.returncode}; see {log}"}, open(out, "w"))
    r = json.load(open(out))
    print(f"  done in {time.time()-t0:.0f}s: {'ok' if r.get('ok') else 'FAILED: ' + str(r.get('error'))[:200]}")
    return r

## 3. Run — each engine in a fresh process on an idle GPU

In [ ]:
from transformers import AutoConfig
cfg = AutoConfig.from_pretrained(MODEL)
kv_heads = getattr(cfg, "num_key_value_heads", cfg.num_attention_heads)
head_dim = getattr(cfg, "head_dim", cfg.hidden_size // cfg.num_attention_heads)
KV_BYTES = 2 * cfg.num_hidden_layers * kv_heads * head_dim * 2 * KV_TOKENS
print(f"KV budget: {KV_TOKENS} tokens = {KV_BYTES/2**20:.0f} MiB")

for variant in ("asis", "no_graphs"):
    print(f"\n▶ repo / {variant}")
    run_proc([sys.executable, f"{WORKDIR}/bench4_repo.py", "--model", MODEL, "--variant", variant,
              "--kv-tokens", str(KV_TOKENS), "--max-active", str(MAX_ACTIVE),
              "--gen-tokens", str(GEN_TOKENS), "--runs", str(RUNS),
              "--out", f"{WORKDIR}/results/repo_{variant}.json"],
             f"{WORKDIR}/logs/repo_{variant}.log", f"{WORKDIR}/results/repo_{variant}.json", env_repo=True)

print("\n▶ naive HF generate")
run_proc([sys.executable, f"{WORKDIR}/bench4_naive.py", "--model", MODEL,
          "--gen-tokens", str(GEN_TOKENS), "--runs", str(RUNS),
          "--out", f"{WORKDIR}/results/naive_hf.json"],
         f"{WORKDIR}/logs/naive.log", f"{WORKDIR}/results/naive_hf.json", env_repo=True)

if RUN_VLLM:
    print("\n▶ vLLM (in-process, matched KV)")
    run_proc([VLLM_PY, f"{WORKDIR}/bench4_vllm.py", "--model", MODEL, "--kv-bytes", str(KV_BYTES),
              "--max-model-len", str(MAX_MODEL_LEN), "--max-active", str(MAX_ACTIVE),
              "--gen-tokens", str(GEN_TOKENS), "--runs", str(RUNS),
              "--out", f"{WORKDIR}/results/vllm.json"],
             f"{WORKDIR}/logs/vllm.log", f"{WORKDIR}/results/vllm.json", env_repo=False)

## Part A — metrics and memory

In [ ]:
import glob
import pandas as pd
import matplotlib.pyplot as plt

R = {}
for path in sorted(glob.glob(f"{WORKDIR}/results/*.json")):
    r = json.load(open(path))
    if r.get("ok"):
        R[r["engine"]] = r
    else:
        print("FAILED", path, r.get("error"), str(r.get("traceback", ""))[-600:])
ORDER = [e for e in ("naive_hf", "repo_no_graphs", "repo_asis", "vllm") if e in R]

rows = []
for e in ORDER:
    r, s = R[e], R[e]["speed"]
    rows.append({"engine": e, "startup s": r.get("startup_s"),
                 "TTFT@512 ms": round(s["ttft_512_ms"], 1), "TPOT ms": round(s["tpot_ms"], 2),
                 "single-stream tok/s": round(s["single_stream_tok_s"], 1),
                 **{f"B{b} tok/s": round(s["batched_tok_s"].get(b, float("nan")), 0) for b in ("1", "4", "8", "16")},
                 "resident MiB": r.get("resident_mib"), "peak MiB": r.get("peak_mib"),
                 "lazy captures": s.get("lazy_graph_captures", "")})
metrics = pd.DataFrame(rows).set_index("engine")
print("=== metrics (T4, Qwen3-0.6B FP16; vLLM in-process) ===")
display(metrics)
metrics.to_csv(f"{WORKDIR}/metrics.csv")

print("\n=== this engine: memory by stage (MiB above idle GPU) — the post-fix graph numbers ===")
mem_rows = []
for e in ("repo_no_graphs", "repo_asis"):
    if e not in R: continue
    a, g = R[e]["attribution_mib"], R[e]["graph_geometry"]
    mem_rows.append({"engine": e, "context": a["cuda_context"], "weights": a["weights"],
                     "KV pool+buffers": a["kv_pool_and_buffers"], "graph pool": a["graph_pool"],
                     "outside allocator": a["outside_allocator_warmup"],
                     "resident": R[e]["resident_mib"], "peak": R[e]["peak_mib"],
                     "graphs d/p/f": f"{g['graphs']}/{g['prefill_graphs']}/{g['fused_graphs']}",
                     "widest rows x ctx": "x".join(map(str, g["widest_captured_rows_x_context"])),
                     "largest gather MiB": g["largest_gather_mib_per_layer"],
                     "dummy tokens": g["dummy_row_tokens"], "warm-up s": R[e]["warmup_s"]})
mem = pd.DataFrame(mem_rows).set_index("engine")
display(mem)
if "vllm" in R:
    print(f"vLLM resident {R['vllm']['resident_mib']} MiB (after load {R['vllm']['resident_after_load_mib']}), "
          f"peak {R['vllm']['peak_mib']} MiB, KV matched: {R['vllm']['config']['kv_capacity_matched']}")
if "naive_hf" in R:
    print(f"naive resident {R['naive_hf']['resident_mib']} MiB, peak {R['naive_hf']['peak_mib']} MiB (DynamicCache, unbounded)")

fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
ax = axes[0]
for e in ORDER:
    bt = R[e]["speed"]["batched_tok_s"]
    xs = sorted(int(k) for k in bt); ax.plot(xs, [bt[str(x)] for x in xs], "o-", label=e)
ax.set_xticks([1, 4, 8, 16]); ax.set_xlabel("concurrent requests"); ax.set_ylabel("output tok/s")
ax.set_title("Batched throughput (256→128)"); ax.legend(fontsize=8); ax.grid(alpha=.3)
ax = axes[1]
ax.bar(ORDER, [R[e]["speed"]["ttft_512_ms"] for e in ORDER]); ax.set_ylabel("ms"); ax.set_title("TTFT, 512-token prompt")
ax.tick_params(axis="x", rotation=20)
ax = axes[2]
ax.bar([e for e in ORDER], [R[e]["resident_mib"] for e in ORDER], label="resident")
ax.bar([e for e in ORDER], [R[e]["peak_mib"] - R[e]["resident_mib"] for e in ORDER],
       bottom=[R[e]["resident_mib"] for e in ORDER], label="peak above resident", alpha=.5)
ax.set_ylabel("MiB"); ax.set_title("GPU memory (NVML, above idle)"); ax.legend(fontsize=8)
ax.tick_params(axis="x", rotation=20)
fig.tight_layout(); fig.savefig(f"{WORKDIR}/metrics.png", dpi=130); plt.show()

## Part B — where a step's device time goes, by transformer sub-block

`attention` = the attention kernels plus KV writes/gathers and their softmax; `gemm_ffn` /
`gemm_attn_proj` / `gemm_lm_head` = GEMM time apportioned by shape; `norm`, `rope`,
`activation`, `sampling`, `embedding`, `memcpy`, `elementwise`, `other` by kernel name.
`wall − device` is host time the GPU spent idle: launch overhead, Python, scheduler. For a
graphed engine it should be small and `graph launches` should be non-zero.

In [ ]:
CLASS_ORDER = ["gemm_ffn", "gemm_attn_proj", "gemm_lm_head", "gemm_other", "gemm_unattributed",
               "attention", "norm", "rope", "activation", "embedding", "sampling", "memcpy",
               "elementwise", "other"]

def breakdown_table(phase):
    rows = []
    for e in ORDER:
        p = R[e]["profile"].get(phase)
        if not p: continue
        row = {"engine": e, "wall ms": round(p.get("wall_ms", float("nan")), 2),
               "device ms": round(p["device_ms_total"], 2),
               "wall − device": round(p.get("wall_ms", float("nan")) - p["device_ms_total"], 2),
               "kernel launches": p["kernel_launches"], "graph launches": p["graph_launches"]}
        for c in CLASS_ORDER:
            v = p["device_ms_by_class"].get(c, 0.0)
            if v: row[c] = round(v, 3)
        rows.append(row)
    return pd.DataFrame(rows).set_index("engine").fillna(0.0)

for phase, title in (("prefill_512", "PREFILL — one 512-token prompt → 1 token"),
                     ("decode_step_b8_ctx512", "DECODE — one step at batch 8, ~512-token contexts (33-run minus 1-run, ÷ 32)")):
    print(f"\n=== {title} (ms) ===")
    t = breakdown_table(phase); display(t); t.to_csv(f"{WORKDIR}/breakdown_{phase}.csv")

if "repo_asis" in R and "decode_one_step_exact" in R["repo_asis"]["profile"]:
    p = R["repo_asis"]["profile"]["decode_one_step_exact"]
    print("\nrepo_asis cross-check, one exact step(): "
          f"wall {p['wall_ms']:.2f} ms, device {p['device_ms_total']:.2f} ms, "
          f"launches {p['kernel_launches']}, graph launches {p['graph_launches']}")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, phase, title in zip(axes, ("prefill_512", "decode_step_b8_ctx512"), ("prefill, 512 tokens", "decode step, batch 8")):
    bottoms = [0.0] * len(ORDER)
    for c in CLASS_ORDER:
        vals = [R[e]["profile"][phase]["device_ms_by_class"].get(c, 0.0) for e in ORDER]
        if not any(vals): continue
        ax.bar(ORDER, vals, bottom=bottoms, label=c)
        bottoms = [b + v for b, v in zip(bottoms, vals)]
    ax.plot(ORDER, [R[e]["profile"][phase].get("wall_ms", 0) for e in ORDER], "k_", markersize=26, label="wall")
    ax.set_title(f"device time by sub-block — {title}"); ax.set_ylabel("ms"); ax.tick_params(axis="x", rotation=20)
axes[0].legend(fontsize=7, ncol=2)
fig.tight_layout(); fig.savefig(f"{WORKDIR}/breakdown.png", dpi=130); plt.show()

## Reading it

- **Memory.** `repo_asis − repo_no_graphs` is the whole price of CUDA graphs on this engine,
  split into the private pool and the executables outside the allocator. `graphs d/p/f`,
  `widest rows x ctx` and `largest gather` are the reachable-shape fix's own numbers — the
  gather should be 64 MiB and the widest capture ≤ pool tokens. Compare `resident` against
  vLLM at the same 16,384-token KV.
- **Decode.** The GEMM rows are the weight read; on a bandwidth-bound card they should be the
  same size for all three engines, because the bytes are. Where engines differ is `attention`,
  the small kernels, and `wall − device`. A large `wall − device` with many `kernel launches`
  and no `graph launches` is the naive engine's signature; the two graphed engines should
  show the opposite.
- **Prefill.** Compute-bound; here kernel choice matters and `attention` is where the engines
  diverge.
- **Caveats.** Kernel classification is by name and GEMM apportioning by shape — a kernel
  with an unfamiliar name lands in `other`; check it is small. vLLM is in-process. The
  profiler itself adds host overhead, so `wall` here is not a latency claim — the metrics
  table is.

In [ ]:
summary = {"gpu": next(iter(R.values()))["gpu"] if R else None,
           "metrics": {e: {"speed": R[e]["speed"], "resident_mib": R[e]["resident_mib"],
                           "peak_mib": R[e]["peak_mib"], "startup_s": R[e].get("startup_s"),
                           "versions": R[e].get("versions")} for e in ORDER},
           "memory_attribution": {e: {"attribution_mib": R[e]["attribution_mib"],
                                      "graph_geometry": R[e].get("graph_geometry")}
                                  for e in ORDER if e.startswith("repo")},
           "breakdown": {e: R[e]["profile"] for e in ORDER}}
json.dump(summary, open(f"{WORKDIR}/part4_summary.json", "w"), indent=2, default=str)
print(json.dumps({e: {"resident": R[e]["resident_mib"], "B16": round(R[e]["speed"]["batched_tok_s"].get("16", 0)),
                      "TTFT": round(R[e]["speed"]["ttft_512_ms"], 1)} for e in ORDER}, indent=1))
!cd {WORKDIR} && zip -qr /content/part4_results.zip results logs *.csv *.png part4_summary.json; ls -la /content/part4_results.zip